# Step 1: PDF ingestion (27-5-26)

## pdf_extractor.py

### A. text, table, image extraction with pdfplumber

In [ ]:
# pip show pillow - to save images into .png (It's the standard Python image library. Lightweight)

In [ ]:
import pdfplumber
import os
from langchain_core.documents import Document
from PIL import Image
from io import BytesIO
import shutil
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
DATA_FOLDER = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod/"

def loader_doc():
    text_doc = []
    table_doc =[]
    image_doc = []

    for file in os.listdir(DATA_FOLDER):
        if file.endswith(".pdf"):
            file_name = file.replace(".pdf", "")
            with pdfplumber.open(f"{DATA_FOLDER}/{file}") as pdf:
                
                if os.path.exists("images"):
                    shutil.rmtree("images")                
                os.makedirs("images", exist_ok=True)
                for page in pdf.pages:
                    if not page:
                        continue
                    else:
                        texts = page.extract_text()
                        
                        text_doc.append(Document(
                                page_content=texts,
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "text",
                                        "source_file": file
                                        # "image_path": {image_path} # image only
                                        }
                                    ))
                        tables = page.extract_tables()
                        for i, table in enumerate(tables):
                            table_doc.append(Document(
                                page_content=str(table),
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "table",
                                        "source_file": file
                                        # "image_path": {image_path} # image only
                                        }
                                    ))
                        images = [img for img in page.images if img["srcsize"][0] >= 100 and img["srcsize"][1] >= 100]                        
                        for i, img in enumerate(images):
                            try:
                                raw_bytes = img["stream"].get_data()
                                pil_image = Image.open(BytesIO(raw_bytes))
                                if pil_image.mode == "CMYK":
                                    pil_image = pil_image.convert("RGB")
                                image_path = pil_image.save(f"images/page_{page.page_number}_image_{i}.png")
                                image_doc.append(Document(
                                    page_content=f"images/page_{page.page_number}_image_{i}.png",
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "image_description",
                                        "source_file": file,
                                        "image_path": image_path # image for only
                                        }
                                    ))
                            except Exception as e:
                                print(f"Skipped image on page {page.page_number}: {e}")
                                continue

    # print(len(text_doc))
    # print(len(table_doc))
    # print(len(image_doc))
    return text_doc, table_doc, image_doc
                        


In [ ]:
# loader_doc() #-- testing

In [ ]:
text_doc, table_doc, image_doc = loader_doc()
# print(len(text_doc))

In [ ]:
# print(type(text_doc[0]))
# print(text_doc[0])

### B. Chunking (Chunker.py)

In [ ]:
def chunker(text_doc):
    splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    chunks = splitter.split_documents(text_doc)
    len(chunks)
    print(chunks[0].page_content)
    print(chunks[0].metadata)
    return chunks

In [ ]:
# chunker(text_doc)

chunks = chunker(text_doc)
print(len(chunks))

In [ ]:
table_chunks = chunker(table_doc)
print(len(table_chunks))

### C. image_processor (image_processor.py)- will get context of the mage that we extracted from  the pdf.

In [9]:
import base64
from openai import OpenAI
import pickle

In [ ]:
def image_processor(image_doc):
    client = OpenAI()
    for i, img in enumerate(image_doc):
        try:
                with open(img.page_content, "rb") as f:
                    image_data = base64.b64encode(f.read()).decode("utf-8")              

                response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{image_data}"
                                }
                            },
                            {
                                "type": "text",
                                "text": """You are analyzing a BMW service manual image, be specific and technical so the description is useful for a technician 
                                searching for information. look for any numbers, labels and text, what that images shows, check for symbols and indicators."""
                            }
                        ]
                    }
                ]
                )
            
                description = response.choices[0].message.content
                img.page_content = description
                print(f"Processing {i+1}/{len(image_doc)}: {img.metadata['page_number']}") #  progress counter inside the loop
        except Exception as e:
                print(f"Skipped {img.page_content}: {e}")
                continue
    
    with open("processed_images.pkl", "wb") as f:
            pickle.dump(image_doc, f)
    return image_doc

In [ ]:
# testing
# processed_images = image_processor(image_doc[10:13])
processed_images = image_processor(image_doc)
# print(processed_images[0].page_content)
# print(len(processed_images))

In [ ]:
# testing the file description got from image_processor which we have saved loacally so we dont have to call the LLM again for all 375 images its costly.
with open("processed_images.pkl", "rb") as f:
    processed_images = pickle.load(f)

print(len(processed_images))
print(processed_images[10].page_content)

In [ ]:
# count how many still have file paths (skipped)
skipped = [img for img in processed_images if img.page_content.startswith("images/")]
print(f"Skipped: {len(skipped)}")
print(f"Processed: {len(processed_images) - len(skipped)}")

In [ ]:
# testing
# print(processed_images[50].page_content)
# print(processed_images[50].metadata)

In [ ]:
#check if chunking needed for the image description. But Most image descriptions are short — 100-300 words. Well under your 2000 character chunk size.So in most cases no chunking needed.
image_chunks = chunker(processed_images)
print(len(image_chunks)) # lenth check

### D. Vectore_store (Vectore_store.py) : storing all embadded vectors in VectorDB

In [ ]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
def vector_store(chunks, collection_name, persist_dir='./BMW_RAG_db'):
    embedding_model=OpenAIEmbeddings(model='text-embedding-3-small')
    vector_store=Chroma.from_documents(
        documents=chunks,
        collection_name=collection_name,
        embedding=embedding_model,
        persist_directory=persist_dir,
        collection_metadata={"hnsw:space":'cosine'}
    )

    print(f"Collection '{collection_name}' created/updated with {len(chunks)} chunks.")

    return vector_store

In [ ]:
# we send chunks to vectore store from here then storing them back into variable to pass them further
text_store = vector_store(chunks + image_chunks, "text_chunks")
table_store = vector_store(table_chunks, "table_chunks")

In [1]:
# **** this is temparary to use vectdb vectorDB locally instade recreate thenm everytime of the run. **** 

import chromadb
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

client_db = chromadb.PersistentClient(path="./BMW_RAG_db")

text_store = Chroma(
    client=client_db,
    collection_name="text_chunks",
    embedding_function=embedding_model
)

table_store = Chroma(
    client=client_db,
    collection_name="table_chunks",
    embedding_function=embedding_model
)

In [ ]:
# **** this is temparary to use vectdb vectorDB locally instade recreate thenm everytime of the run. **** 
print(text_store._collection.count())   # should print 1101
print(table_store._collection.count())  # should print 1

In [ ]:
# testing -
# 1.
# results = text_store.similarity_search("parking brake", k=3)
# for r in results:
#     print(r.page_content)
#     print(r.metadata)
#     print("---")
# # 2.
# results = text_store.similarity_search("steering wheel controls", k=3)
# for r in results:
#     print(r.page_content[:200])
#     print(r.metadata)
#     print("---")

# ======== Step 1 complete ===============================

# Step 2 - Query classifier (LangGraph node : Date - 29-2026)

## A. state.py

#### IMP ---- we have consider the context in architecture but not using them in project as we have only 1 manual
####(Vehicle context = (make, model, year, engine))

In [2]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
import operator
from openai import OpenAI

In [ ]:
class Citation(TypedDict):
    page: int
    section: str
    source: str

class AgentState(TypedDict):
    user_type: str
    query: str
    intent: str
    retrieved_chunks: Annotated[list[dict], operator.add]  # ✅ merge parallel writes
    confidence_score: float
    conversation_history: Annotated[list[BaseMessage], operator.add] 
    citations: Annotated[list[Citation], operator.add]  
    image_paths: Annotated[list[str], operator.add]  

### B. Classifier to classify from which vectoreDB need to find An answer for the user query?
    Tables and text are structurally different. If someone asks: -
    "what does the oil warning light mean" → answer is in prose text, search text_chunks
    "what is the torque spec for front wheel bolts" → answer is in a table row, search table_chunks
    2. What intent means
    Intent = what the user is trying to do, not the literal words they used.
    Same underlying need, different phrasings:
    "there's a yellow circle on my dash"
    "warning light came on"              →  intent for all three is : warning_light
    "my car is showing some symbol"

### classifier.py

In [4]:
client = OpenAI()
def classifier (state):
    VALID_ROUTES = ["text", "table", "both", "unknown"]
    response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                {"role": "system", "content": """You are a routing assistant for a BMW service manual.
                        Given a user query, decide which data source to search.

                        Return only one of-
                        text: descriptive information, procedures, explanations, warnings, 
                        fluid types and specifications (e.g. which oil to use), 
                        component descriptions, operating instructions
                 
                        table: numerical specs, exact measurements, torque values (e.g. 25 Nm), 
                        service intervals (e.g. every 10,000 miles), scheduled maintenance dates, 
                        fault/error codes, fluid capacities in exact quantities
  
                        both: multiple symptoms, diagnostic queries, anything where 
                        the answer needs both an explanation AND a spec value, 
                        strange noises, car not starting, complex issues
                 
                        unknown: ONLY use this for queries that are completely 
                        outside vehicle service manual scope — prices, dealer 
                        locations, purchase advice, non-automotive topics.
                        Any query about vehicle symptoms, warnings, maintenance, 
                        repairs, or parts belongs to text/table/both."""},
                {"role": "user", "content": state["query"]}
                ]
                
                )
    
    intent = response.choices[0].message.content.strip().lower()
    if intent not in VALID_ROUTES:
        intent = "unknown"
    return {"intent": intent}
                    

In [29]:
# testing :
# Test queries
queries = [
    "there is a yellow circle on my dashboard",
    "what is the torque spec for front wheel bolts",
    "when is my next oil change due",
    "my car is making a strange noise and won't start",
    "tell me oprating element on steering wheel",
    "tell me price of head light"
]

for q in queries:
    state = {"query": q}
    result = classifier(state)
    print(f"Query: {q}")
    print(f"Intent: {result['intent']}")
    print("---")

Query: there is a yellow circle on my dashboard
Intent: text
---
Query: what is the torque spec for front wheel bolts
Intent: table
---
Query: when is my next oil change due
Intent: table
---
Query: my car is making a strange noise and won't start
Intent: both
---
Query: tell me oprating element on steering wheel
Intent: text
---
Query: tell me price of head light
Intent: unknown
---


In [ ]:
# 1. testing
test_state = {
    "user_type": "owner",
    "query": "there is a yellow warning light on my dashboard",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])

In [ ]:
# 2. Now test the diagnostic case — this is the one that triggers parallel execution of both retrievers:
test_state = {
    "user_type": "technician",
    "query": "my car is making a strange noise and won't start",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])

In [5]:
from langgraph.graph import StateGraph, END

def route_intent(state):
    intent = state["intent"]
    if intent == "both":
        return ["text_retriever", "table_retriever"]
    elif intent == "table":
        return "table_retriever"
    elif intent == "unknown":
        return "unknown_handler"
    else:
        return "text_retriever"

### empty chunks handler node

In [6]:
def unknown_handler(state):
    return {
        "conversation_history": state["conversation_history"] + [
            AIMessage(content="I couldn't find relevant information in the BMW service manual for your query. Please consult a certified BMW technician or visit your nearest service center.")
        ]
    }

# ======== Step 2 complete ===============================

# Step 3 - Create retivers/empty chunk handling (30-5-26)

## text_retriever.py

In [7]:
def text_retriever(state):
    results = text_store.similarity_search_with_score(state["query"], k=5)
    chunks = []  
    for doc, score in results:
        chunks.append({ 
            "content": doc.page_content,
            "metadata": doc.metadata,
            "score": score
        })
    return {"retrieved_chunks": chunks}

In [ ]:
# testing 
test_state = {
    "user_type": "owner",
    "query": "what does the oil warning light mean",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])
print("Chunks retrieved:", len(result["retrieved_chunks"]))
for chunk in result["retrieved_chunks"]:
    print("Score:", chunk["score"])
    print("Content preview:", chunk["content"][:100])
    print("---")

### table_retriever.py

In [8]:
def table_retriever(state):
    results = table_store.similarity_search_with_score(state["query"], k=5)
    chunks = []  
    for doc, score in results:
        chunks.append({ 
            "content": doc.page_content,
            "metadata": doc.metadata,
            "score": score
        })
    return {"retrieved_chunks": chunks}

In [49]:
# testing
test_state = {
    "user_type": "technician",
    "query": "what is the torque spec for front wheel bolts",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])
print("Chunks retrieved:", len(result["retrieved_chunks"]))
for chunk in result["retrieved_chunks"]:
    print("Score:", chunk["score"])
    print("Content preview:", chunk["content"][:100])
    print("---")

Intent: torque_spec
Chunks retrieved: 1
Score: 0.8863277435302734
Content preview: [['', '']]
---


In [47]:
# testing
queries = [
    ("there is a yellow circle on my dashboard", "text route"),
    ("what is the torque spec for front wheel bolts", "table route"),
    ("my car is making a strange noise and won't start", "both route"),
    ("tell me the price of a headlight", "unknown route")
]

for query, expected in queries:
    result = app.invoke({
        "user_type": "owner",
        "query": query,
        "intent": "",
        "retrieved_chunks": [],
        "confidence_score": 0.0,
        "conversation_history": [],
        "citations": [],
        "image_paths": []
    })
    print(f"Query: {query}")
    print(f"Expected: {expected}")
    print(f"Intent: {result['intent']}")
    print(f"Chunks: {len(result['retrieved_chunks'])}")
    print("---")

Query: there is a yellow circle on my dashboard
Expected: text route
Intent: text
Chunks: 5
---
Query: what is the torque spec for front wheel bolts
Expected: table route
Intent: table
Chunks: 1
---
Query: my car is making a strange noise and won't start
Expected: both route
Intent: both
Chunks: 6
---
Query: tell me the price of a headlight
Expected: unknown route
Intent: unknown
Chunks: 0
---


# ======== Step 3 complete ===============================

# Step 4 : Multi-turn diagnostic conversation + memory, connfidence and citation

## A. conversation.py

In [9]:
# from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
# import operator
from openai import OpenAI

In [10]:
client = OpenAI()

def conversation(state):
    if not state["retrieved_chunks"]:
        return {
            "conversation_history": state["conversation_history"] + [
                AIMessage(content="I couldn't find relevant information in your BMW manual for this query. Please consult a certified BMW technician.")
            ]
        }

    context = "\n\n".join([
        f"Page {chunk['metadata'].get('page_number', 'unknown')}:\n{chunk['content']}"
        for chunk in state["retrieved_chunks"]
    ])

    user_type = state["user_type"]
    citation_text = "\n".join([
    f"- Page {c['page']}, Section {c['section']}"
    for c in state["citations"]])

    if user_type == "owner":
        system_prompt = f"""You are a BMW service manual assistant helping a car owner.
                        Use simple, non-technical language. Avoid jargon.
                        Always recommend visiting a certified BMW service center for repairs.
                        Base your answer only on the provided manual context.
                        Reference these manual sections: {citation_text}
                        Keep the response concise and under 150 words."""

    else:
        system_prompt = f"""You are a BMW service manual assistant helping a certified technician.
                        Use precise technical language. Include specifications, torque values, and part references where available.
                        Always cite the page number from the manual context in your response.
                        Base your answer only on the provided manual context.
                        Reference these manual sections: {citation_text}"""
                        

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            *state["conversation_history"],
			
            {"role": "user", "content": f"{state['query']}\n\nContext:\n{context}"}
        ]
    )

    answer = response.choices[0].message.content.strip()
    
    if state["confidence_score"] > 0.7:
        answer += "\n\n⚠️ Note: This answer is based on limited matches from the manual. Please verify with a certified BMW technician."

    return {
        "conversation_history": state["conversation_history"] + [
            HumanMessage(content=state["query"]),
            AIMessage(content=answer)
        ]
    }

In [ ]:
# testing 1 - an empty chunks
test_state = {
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "text",
    "retrieved_chunks": [[(Document(id='ae2473c1-d090-4e0b-92f9-676ea1e7a1bf', metadata={'chunk_type': 'text', 'source_file': 'bmw_manual.pdf', 'page_number': 417, 'section': 'page_417'}, page_content='Seite 417\nOperating fluids MOBILITY\nPetrol engine\nNOTICE\nACEA C2.\nUsing the wrong engine oil can result in en-\ngine malfunctions and damage. There is a ACEA C3.\nrisk of material damage. When selecting the\nACEA C5.\nengine oil, make sure that it is the correct oil\nspecification.\nDiesel engine\nSuitable engine oil grades ACEA C2.\nEngine oil with the following oil specification ACEA C3.\ncan be topped up:\nPetrol engine with exhaust gas particulate Viscosity classes\nfilter When selecting an engine oil, make sure that\nthe engine oil belongs to one of the following\nBMW Longlife-12 FE.\nviscosity classes:\nBMW Longlife-17 FE+.\nPetrol engine\nBMW Longlife-19 FE.\nSAE 0W-12.\nBMW Longlife-22 FE++.\nSAE 0W-20.\nSAE 0W-30.\nPetrol engine without exhaust gas particu-\nlate filter\nThe viscosity classes SAE 0W-12 and SAE\nBMW Longlife-01 FE. 0W-20 are not suitable for the 60i petrol en-\ngine.\nBMW Longlife-17 FE+.\nDiesel engine\nBMW Longlife-22 FE++.\nSAE 0W-30.\nThe oil specifications BMW Longlife-17 FE+\nand BMW Longlife-22 FE+ are not suitable for Viscosity classes with a high viscosity grade\nuse with 60i petrol engines. can increase fuel consumption.\nDiesel engine Further information on suitable engine oil\nspecifications and viscosity classes can be ob-\nBMW Longlife-12 FE.\ntained from an authorised Service Partner or\nBMW Longlife-19 FE. another qualified Service Partner or a specialist\nworkshop.\nAlternative engine oil grades\nIf suitable engine oils are not available, up to\n1 litre, 2 pints of an engine oil with the following\noil specification can be used for topping up:\n417\nOnline Edition for Part no. 01405B64643 - II/25'), 0.3853793144226074), (Document(id='57053802-4d79-4e47-b9fb-2eda0fc74d98', metadata={'page_number': 34, 'section': 'page_34', 'chunk_type': 'image_description', 'source_file': 'bmw_manual.pdf'}, page_content='The image shows an engine oil filler cap, commonly found in a BMW vehicle. The cap has an illustration of an oil can symbol, which indicates its purpose for adding engine oil. There is an arrow pointing counterclockwise, suggesting the direction to turn the cap to open it. No numbers, text, or additional labels are visible aside from the symbols and arrow. This cap is typically located on the top of the engine and is a crucial component for maintaining proper engine lubrication by allowing oil to be added easily.'), 0.4909687042236328), (Document(id='2dbefea3-ed87-408f-a24e-0e5456afc829', metadata={'source_file': 'bmw_manual.pdf', 'chunk_type': 'image_description', 'page_number': 77, 'section': 'page_77'}, page_content='The image shows a technical diagram of a component that appears to be an oil or fluid filter used in BMW vehicles. It is viewed from the top, with the primary focus on the circular filter element. The surrounding structure includes a housing that may be integrated into the engine or fluid system.\n\n1. **Central Component**: The dominant round element is likely the filter medium itself. It is enclosed by a circular housing which would be designed to fit into its designated slot within the vehicle.\n\n2. **Housing Details**: To one side, there is an L-shaped extension which is likely part of the filter housing. This part might include connection points or clips for securing the filter in place.\n\n3. **Seal and O-Rings**: The presence of a darker, possibly rubber ring around the circular component suggests a seal or an O-ring, crucial for preventing leaks.\n\n4. **Material Design**: The housing appears to be composed of light-colored material, potentially plastic or metal coated with a protective layer to withstand engine temperatures and fluid corrosion.\n\n5. **Possible Labels or Text**: There are no visible numbers or text on the housing visible in this particular angle of the diagram.\n\n6. **Connections and Ports**: The openings or tabs seen on the right side of the housing may indicate connection points for fluid lines or sensor integration.\n\nThis image would be useful for a technician when identifying the correct filter type and ensuring proper orientation during installation or replacement in a BMW vehicle.'), 0.5000331401824951), (Document(id='70fc082c-82cf-4bcd-8fda-3c02836adba3', metadata={'chunk_type': 'text', 'page_number': 5, 'section': 'page_5', 'source_file': 'bmw_manual.pdf'}, page_content='© 2025 Bayerische Motoren Werke\nAktiengesellschaft\nMunich, Germany\nNot to be reproduced, wholly or in part, without written permission from BMW AG, Munich.\nEnglish ID8 II/25, -\nPrinted on environmentally friendly paper, bleached without chlorine, suitable for recycling.\n5\nOnline Edition for Part no. 01405B64643 - II/25'), 0.5020538568496704), (Document(id='1de56dc8-eda4-486a-b5e2-6f9c561abf38', metadata={'page_number': 1, 'section': 'page_1', 'chunk_type': 'text', 'source_file': 'bmw_manual.pdf'}, page_content="Content A-Z\nOWNER'S HANDBOOK.\nBMW X5.\nOnline Edition for Part no. 01405B64643 - II/25"), 0.517418622970581)]],  # test empty path first
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = text_store.similarity_search_with_score("recommended oil BMW", k=5)
print(result)

[(Document(id='ae2473c1-d090-4e0b-92f9-676ea1e7a1bf', metadata={'chunk_type': 'text', 'source_file': 'bmw_manual.pdf', 'page_number': 417, 'section': 'page_417'}, page_content='Seite 417\nOperating fluids MOBILITY\nPetrol engine\nNOTICE\nACEA C2.\nUsing the wrong engine oil can result in en-\ngine malfunctions and damage. There is a ACEA C3.\nrisk of material damage. When selecting the\nACEA C5.\nengine oil, make sure that it is the correct oil\nspecification.\nDiesel engine\nSuitable engine oil grades ACEA C2.\nEngine oil with the following oil specification ACEA C3.\ncan be topped up:\nPetrol engine with exhaust gas particulate Viscosity classes\nfilter When selecting an engine oil, make sure that\nthe engine oil belongs to one of the following\nBMW Longlife-12 FE.\nviscosity classes:\nBMW Longlife-17 FE+.\nPetrol engine\nBMW Longlife-19 FE.\nSAE 0W-12.\nBMW Longlife-22 FE++.\nSAE 0W-20.\nSAE 0W-30.\nPetrol engine without exhaust gas particu-\nlate filter\nThe viscosity classes SAE 0

In [ ]:
# testing 1 actual owners test case
raw = text_store.similarity_search_with_score("recommended oil BMW", k=5)

test_state = {
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "text",
    "retrieved_chunks": [
        {"content": doc.page_content, "metadata": doc.metadata, "score": score}
        for doc, score in raw
    ],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = conversation(test_state)
print(result)

{'conversation_history': [HumanMessage(content='What is the recommended oil for my BMW?', additional_kwargs={}, response_metadata={}), AIMessage(content="For your BMW's petrol engine, it's important to use the correct type of engine oil to keep it running smoothly and avoid damage.\n\nIf your BMW petrol engine has an exhaust gas particulate filter, you should use oil with specifications like BMW Longlife-12 FE, BMW Longlife-17 FE+, BMW Longlife-19 FE, or BMW Longlife-22 FE++. The suitable viscosity classes for these oils are SAE 0W-12, SAE 0W-20, and SAE 0W-30.\n\nFor petrol engines without an exhaust gas particulate filter, you can also use BMW Longlife-01 FE or BMW Longlife-17 FE+. Just remember, SAE 0W-12 and SAE 0W-20 are not recommended for the 60i petrol engine.\n\nIf your BMW has a diesel engine, suitable oil would be BMW Longlife-12 FE, BMW Longlife-19 FE, BMW Longlife-17 FE+, or BMW Longlife-22 FE++. The recommended viscosity class for diesel engines is SAE 0W-30.\n\nIf you're

# ======== Step 4 complete ===============================

# Step 5 : Confidance_score + citation 

## confidence.py

In [11]:
def confidence_score (state):
    chunks = state["retrieved_chunks"]
    if not chunks:
        return {"confidence_score": 0.0, "citations": []}
    avg_score = sum(chunk["score"] for chunk in chunks) / len(chunks)
        
    citation = []
    for chunk in chunks:
        citation.append({
                "page": chunk["metadata"].get("page_number"),
                "section": chunk["metadata"].get("section"),
                "source": chunk["metadata"].get("source_file")
            })              
    return {"confidence_score" : avg_score, "citations" : citation}
   

In [ ]:
# testing
raw = text_store.similarity_search_with_score("recommended oil BMW", k=5)

test_state = {
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "text",
    "retrieved_chunks": [
        {"content": doc.page_content, "metadata": doc.metadata, "score": score}
        for doc, score in raw
    ],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = confidence_score(test_state)
print(result)

{'confidence_score': 0.47918167114257815, 'citations': [{'page': 417, 'section': 'page_417', 'source': 'bmw_manual.pdf'}, {'page': 34, 'section': 'page_34', 'source': 'bmw_manual.pdf'}, {'page': 77, 'section': 'page_77', 'source': 'bmw_manual.pdf'}, {'page': 5, 'section': 'page_5', 'source': 'bmw_manual.pdf'}, {'page': 1, 'section': 'page_1', 'source': 'bmw_manual.pdf'}]}


# ======== Step 5 complete ===============================

# Step 6 : AzureOpenAI() swap

#### For now, staying on OpenAI() is fine. The swap to AzureOpenAI() is a Step 6 task — just a client swap, not an architectural change.Want to skip Step 6 for now and move to Step 7

# ======== Step 6 complete ===============================

# Step 7: FastAPI + Docker 

## main.py

In [ ]:
import uvicorn
from api.app import api

if __name__ == "__main__":
    uvicorn.run(api, host="0.0.0.0", port=8000, reload=True)


## app.py

In [5]:
from fastapi import FastAPI
from pydantic import BaseModel
from agent.graph import app as agent_app
from agent.state import Citation

api = FastAPI() # FastAPI() creates your web application instance.
sessions: dict = {}

# Request — what client sends
class QueryRequest(BaseModel):
    query: str
    session_id: str
    user_type: str

# Response — what you send back
class QueryResponse(BaseModel):
    answer: str
    citations: list[Citation]
    confidence_score: float

@api.post("/query")
def query(request: QueryRequest):
    result = agent_app.invoke({
    "query": request.query,
    "user_type": request.user_type,
    "conversation_history": sessions.get(request.session_id, []),  # loaded from sessions dict
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "citations": [],
    "image_paths": []
    })
    last_message = result["conversation_history"][-1].content #That line reads the last message from the result. It does not save anything, This READS the last AIMessage content to return as the answer
    sessions[request.session_id] = result["conversation_history"] #This SAVES the full updated history back into sessions
    return QueryResponse(
    answer=last_message,
    citations=result["citations"],
    confidence_score=result["confidence_score"]
)

ModuleNotFoundError: No module named 'agent'

# Graph = includes all steps as it executes the whole conversation loop

In [13]:
graph = StateGraph(AgentState)
graph.add_node('classifier', classifier)
graph.add_node('text_retriever', text_retriever)   
graph.add_node('table_retriever', table_retriever) 
graph.add_node('unknown_handler', unknown_handler)   
graph.add_node("confidence", confidence_score)
graph.add_node("conversation", conversation)

graph.set_entry_point('classifier')
graph.add_conditional_edges(
    "classifier",      # after this node
    route_intent,      # call this function
    {
        "text_retriever": "text_retriever",
        "table_retriever": "table_retriever",
        "unknown_handler" : "unknown_handler"
    }
)
# graph.add_edge('router', 'retriever') 
graph.add_edge('text_retriever', 'confidence')   
graph.add_edge('table_retriever', 'confidence')  
graph.add_edge('unknown_handler', END)
graph.add_edge('confidence', 'conversation') 
graph.add_edge('conversation', END) 


app = graph.compile()

In [14]:
# testing - whole graph after adding confidence_score and Citation
result = app.invoke({
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
})

print(result["conversation_history"][-1].content)
print(result["confidence_score"])
print(result["citations"])


According to the manual, your BMW engine uses specific oil types based on the type of engine. For petrol engines, suitable oils include BMW Longlife-12 FE, BMW Longlife-17 FE+, BMW Longlife-19 FE, and BMW Longlife-22 FE++. If your petrol engine has an exhaust gas particulate filter, SAE 0W-12, SAE 0W-20, and SAE 0W-30 oils are recommended. Diesel engines should use BMW Longlife-12 FE and BMW Longlife-19 FE. When selecting engine oil, ensure it meets the correct specification to avoid engine damage. If you're unsure or need further guidance, it's recommended to visit a certified BMW service center.
0.492783260345459
[{'page': 417, 'section': 'page_417', 'source': 'bmw_manual.pdf'}, {'page': 77, 'section': 'page_77', 'source': 'bmw_manual.pdf'}, {'page': 34, 'section': 'page_34', 'source': 'bmw_manual.pdf'}, {'page': 3, 'section': 'page_3', 'source': 'bmw_manual.pdf'}, {'page': 406, 'section': 'page_406', 'source': 'bmw_manual.pdf'}]


In [ ]:
# project folder architecture check :

import os
for root, dirs, files in os.walk("."):
    # skip hidden and cache folders
    dirs[:] = [d for d in dirs if d not in ['.git', '__pycache__', '.ipynb_checkpoints']]
    level = root.replace(".", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{root}/")
    for file in files:
        print(f"{indent}  {file}")

./
  bmw_manual.pdf
  chunker.ipynb
  pdf_extractor.py
  processed_images.pkl
  project_coding.ipynb
  .\Architecture/
    Architechture_of_project.docx
    bmw_rag_agent_architecture.svg
    Pending things in project as an architecture.docx
  .\BMW_RAG_db/
    chroma.sqlite3
    .\BMW_RAG_db\c698d3ab-bb69-4fe7-aadf-83ad7bf21d61/
      data_level0.bin
      header.bin
      length.bin
      link_lists.bin
    .\BMW_RAG_db\d4e56f4b-ee4f-4ccc-ae20-efa6c58f5c96/
      data_level0.bin
      header.bin
      index_metadata.pickle
      length.bin
      link_lists.bin
  .\images/
    page_101_image_0.png
    page_101_image_1.png
    page_102_image_0.png
    page_103_image_0.png
    page_104_image_0.png
    page_104_image_1.png
    page_104_image_2.png
    page_104_image_3.png
    page_104_image_4.png
    page_105_image_0.png
    page_106_image_0.png
    page_106_image_1.png
    page_107_image_0.png
    page_108_image_0.png
    page_108_image_1.png
    page_108_image_2.png
    page_108_image_

In [ ]:
# !pip freeze > requirements.txt

In [9]:
import nbformat

with open("project_coding.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

for i, cell in enumerate(nb.cells):
    if cell.cell_type == "code":
        first_line = cell.source.split("\n")[0] if cell.source else "(empty)"
        print(f"Cell {i}: {first_line}")

Cell 3: # pip show pillow - to save images into .png (It's the standard Python image library. Lightweight)
Cell 4: import pdfplumber
Cell 5: DATA_FOLDER = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod/"
Cell 6: # loader_doc() #-- testing
Cell 7: text_doc, table_doc, image_doc = loader_doc()
Cell 8: # print(type(text_doc[0]))
Cell 10: def chunker(text_doc):
Cell 11: # chunker(text_doc)
Cell 12: table_chunks = chunker(table_doc)
Cell 14: import base64
Cell 15: def image_processor(image_doc):
Cell 16: # testing
Cell 17: # testing the file description got from image_processor which we have saved loacally so we dont have to call the LLM again for all 375 images its costly.
Cell 18: # count how many still have file paths (skipped)
Cell 19: # testing
Cell 20: #check if chunking needed for the image description. But Most image descriptions are short — 100-300 words. Well under your 2000 character chunk size.So in most cases no chunking needed.
Cell 22: imp